# Del .pt

In [ ]:
import os
import glob

def delete_pt_files(base_dir, models_to_delete, version_start, version_end):
    """
    선택한 모델들의 특정 실험 version 범위에서 모든 .pt 파일을 삭제하는 함수.

    Parameters:
        base_dir (str): 실험 결과가 저장된 기본 디렉토리
        models_to_delete (list): 삭제할 모델 목록 (예: ['DLinear', 'iTransformer'])
        version_start (int): 삭제할 최소 버전 (예: 0)
        version_end (int): 삭제할 최대 버전 (예: 8)
    """
    for model in models_to_delete:
        model_dir = os.path.join(base_dir, model, "KETI_weather_prediction")
        
        if not os.path.exists(model_dir):
            print(f"⚠️ 경고: {model_dir} 경로가 존재하지 않습니다. 스킵합니다.")
            continue
        
        for version in range(version_start, version_end + 1):
            version_dir = os.path.join(model_dir, f"version{version}")
            
            if not os.path.exists(version_dir):
                print(f"⚠️ 경고: {version_dir} 경로가 존재하지 않습니다. 스킵합니다.")
                continue

            # .pt 파일 찾기
            pt_files = glob.glob(os.path.join(version_dir, "*.pt"))
            
            if not pt_files:
                print(f"✅ version{version} 폴더에는 삭제할 .pt 파일이 없습니다.")
                continue
            
            # 삭제 수행
            for pt_file in pt_files:
                try:
                    os.remove(pt_file)
                    print(f"🗑 삭제됨: {pt_file}")
                except Exception as e:
                    print(f"❌ 삭제 실패: {pt_file} - 오류: {e}")

# 사용 예제
base_directory = "../src/saved_model/version2_241107"
models = ["DLinear", "iTransformer", "PatchTST", "TimesNet"]  # 삭제할 모델 선택
version_start = 0
version_end = 4

delete_pt_files(base_directory, models, version_start, version_end)

---

# Plot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error


def load_predictions_and_targets(root_dir, version):
    version_path = os.path.join(root_dir, version)
    output_path = os.path.join(version_path, "TEST_total_outputs.npy")
    target_path = os.path.join(version_path, "TEST_total_targets.npy")

    if not os.path.exists(output_path) or not os.path.exists(target_path):
        print(f"[경고] {version}에서 output 또는 target 파일이 없습니다.")
        return None, None

    pred_dict = np.load(output_path, allow_pickle=True).item()
    target_dict = np.load(target_path, allow_pickle=True).item()

    return pred_dict, target_dict

def plot_prediction_vs_target(
    pred_dict,
    target_dict,
    model_name,
    location,
    hogi=None,
    start=0,
    length=500,
    ylim=None
):
    if hogi:
        key = (model_name, f"{location}_{hogi}")
    else:
        key = (model_name, location)

    if key not in pred_dict or key not in target_dict:
        print(f"[경고] {key}가 예측 또는 타깃 데이터에 존재하지 않습니다.")
        return
    print(pred_dict[key].shape, target_dict[key].shape)
    pred = pred_dict[key].flatten()
    target = target_dict[key].flatten()
    
    # 전체 MSE, MAE, RMSE 계산
    mse = mean_squared_error(target, pred)
    mae = mean_absolute_error(target, pred)
    rmse = np.sqrt(mse)

    print(f"📊 전체 데이터 기준 성능 ({key})")
    print(f"   MSE:  {mse:.4f}")
    print(f"   MAE:  {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")

    # 시각화 범위 자르기
    end = start + length
    pred_slice = pred[start:end]
    target_slice = target[start:end]

    if ylim is None:
        y_min = min(np.min(pred_slice), np.min(target_slice))
        y_max = max(np.max(pred_slice), np.max(target_slice))
        ylim = (y_min - y_min*0.1, y_max + y_max*0.1)

    plt.rcParams.update({'font.size': 10})
    plt.figure(figsize=(16, 8))

    pastel_blue = "#A1C9F4"
    pastel_orange = "#FFB482"

    x_range = np.arange(start, end)
    plt.plot(x_range, pred_slice, label="Prediction", color=pastel_blue, linewidth=2)
    plt.plot(x_range, target_slice, label="Target", color=pastel_orange, linewidth=2)
    
    set_x_range = start+5, end-5
    plt.xlim(set_x_range[0], set_x_range[1])
    plt.ylim(ylim)

    plt.tick_params(axis='both', which='major', labelsize=12)

    title = f"{model_name} - {location}"
    if hogi:
        title += f"_{hogi}"
    title += f"\nMSE: {mse:.3f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}"
    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.grid(False)
    plt.legend(fontsize=11)
    plt.show()
def plot_from_model_version(
    model, version, model_name, location, hogi=None, start=0, length=1000
):
    root_dir = os.path.join("saved_model", "version2_241107", model, "KETI_weather_prediction")
    pred_dict, target_dict = load_predictions_and_targets(root_dir, version)

    if pred_dict is None or target_dict is None:
        return

    plot_prediction_vs_target(
        pred_dict, target_dict, model_name, location, hogi, start=start, length=length
    )

In [ ]:
plot_from_model_version(
    model="DLinear",
    version="version7",
    model_name="ECMWF",
    location="6498",
    hogi=None
)

In [ ]:
plot_from_model_version(
    model="DLinear",
    version="version8",
    model_name="ECMWF",
    location="5375",
    hogi=None
)

In [ ]:
plot_from_model_version(
    model="DLinear",
    version="version10",
    model_name="ECMWF",
    location="4593_006",
    hogi=None
)

---

# Cal metric

In [ ]:
import os
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import yaml

def MAE(pred, true):
    return np.mean(np.abs(pred - true))

def MSE(pred, true):
    return np.mean((pred - true) ** 2)

def RMSE(pred, true):
    return np.sqrt(MSE(pred, true))

def load_config(config_path):
    if not os.path.exists(config_path):
        print(f"[경고] config 파일이 존재하지 않습니다: {config_path}")
        return "unknown", "unknown"

    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    dataset_cfg = config.get("DATASET", {})
    target = dataset_cfg.get("category_name", "unknown")
    seq_len = dataset_cfg.get("seq_len", "unknown")

    return target, seq_len

def evaluate_model_version(model: str, version: str):
    base_path = os.path.join("saved_model", "version2_241107", model, "KETI_weather_prediction", version)
    pred_path = os.path.join(base_path, "TEST_total_outputs.npy")
    target_path = os.path.join(base_path, "TEST_total_targets.npy")
    config_path = os.path.join(base_path, "configs.yaml")

    if not os.path.exists(pred_path) or not os.path.exists(target_path):
        print(f"\n[스킵] {model}/{version} — .npy 파일 없음")
        return

    category_name, seq_len = load_config(config_path)
    print(f"\n📘 [{model}/{version}] | category_name: `{category_name}` | seq_len: {seq_len}")

    pred_dict = np.load(pred_path, allow_pickle=True).item()
    target_dict = np.load(target_path, allow_pickle=True).item()

    mse_list, mae_list, rmse_list = [], [], []

    for key in pred_dict:
        if key not in target_dict:
            print(f"[경고] {key}는 target_dict에 없음 — 스킵")
            continue

        pred = pred_dict[key].flatten()
        target = target_dict[key].flatten()

        mse = MSE(pred, target)
        mae = MAE(pred, target)
        rmse = RMSE(pred, target)

        mse_list.append(mse)
        mae_list.append(mae)
        rmse_list.append(rmse)

    print(f"📊 평균 성능: MSE={np.mean(mse_list):.4f}, MAE={np.mean(mae_list):.4f}, RMSE={np.mean(rmse_list):.4f}")


def evaluate_multiple_versions(model_list, range_start, range_end):
    versions = [f"version{i}" for i in range(range_start, range_end+1)]  # version6 to version11
    for model in model_list:
        for version in versions:
            evaluate_model_version(model, version)

In [ ]:
model_list = ["DLinear", "PatchTST", "iTransformer"]
evaluate_multiple_versions(model_list, 6, 11)

In [ ]:
import pandas as pd
def extract_location(loc_str):
    """
    '4593_001' -> '4593', '5375' -> '5375' 형식으로 변환
    """
    return loc_str.split('_')[0] if '_' in loc_str else loc_str

def evaluate_model_version_by_location(model: str, version: str):
    base_path = os.path.join("saved_model", "version2_241107", model, "KETI_weather_prediction", version)
    pred_path = os.path.join(base_path, "TEST_total_outputs.npy")
    target_path = os.path.join(base_path, "TEST_total_targets.npy")
    config_path = os.path.join(base_path, "configs.yaml")

    if not os.path.exists(pred_path) or not os.path.exists(target_path):
        return None

    category_name, seq_len = load_config(config_path)
    pred_dict = np.load(pred_path, allow_pickle=True).item()
    target_dict = np.load(target_path, allow_pickle=True).item()

    # location 기준 성능 누적
    location_scores = {}
    for key in pred_dict:
        if key not in target_dict:
            continue
        _, loc = key
        loc_id = extract_location(loc)

        pred = pred_dict[key].flatten()
        target = target_dict[key].flatten()

        if loc_id not in location_scores:
            location_scores[loc_id] = {"mse": [], "mae": [], "rmse": []}
        
        location_scores[loc_id]["mse"].append(MSE(pred, target))
        location_scores[loc_id]["mae"].append(MAE(pred, target))
        location_scores[loc_id]["rmse"].append(RMSE(pred, target))

    # 평균 후 DataFrame으로 정리
    records = []
    for loc_id, scores in location_scores.items():
        records.append({
            "loc": loc_id,
            "Window size": seq_len,
            "Model": model,
            "MSE": np.mean(scores["mse"]),
            "MAE": np.mean(scores["mae"]),
            "RMSE": np.mean(scores["rmse"])
        })

    return records

def evaluate_multiple_versions_grouped(model_list, range_start, range_end):
    versions = [f"version{i}" for i in range(range_start, range_end + 1)]
    all_records = []
    for model in model_list:
        for version in versions:
            result = evaluate_model_version_by_location(model, version)
            if result:
                all_records.extend(result)
    return pd.DataFrame(all_records)

# 실행
model_list = ["DLinear", "PatchTST", "TimesNet", "iTransformer"]
df_grouped_result = evaluate_multiple_versions_grouped(model_list, 6, 11)

In [ ]:
import pandas as pd

# 모델 정렬 순서 정의
model_order = ['DLinear', 'PatchTST', 'iTransformer']
model_order_map = {name: i for i, name in enumerate(model_order)}
df = df_grouped_result
# 사용자 지정 모델 순서 열 추가
df['Model_sort'] = df['Model'].map(model_order_map)

# 정렬
df_sorted = df.sort_values(by=['loc', 'Window size', 'Model_sort']).drop(columns='Model_sort')

df_sorted.to_csv("model_performance_by_location.csv", index=False)